# Informe 1.7 – Análisis Big Data con Apache Spark y reflexión de escalabilidad
**Proyecto:** DataSalud Perú – DIRESA La Libertad  |  **Curso:** Base de Datos Avanzadas y Big Data (UPN)

**Dataset:** vacunación COVID-19 del MINSA (`vacunas_covid.csv`, ~1.05 millones de registros, 14 columnas).

**Objetivo:** procesar el dataset con **RDDs, DataFrames y Spark SQL** (los tres objetos), ejecutar operaciones de estadística descriptiva y agregación sobre datos masivos, medir tiempos y compararlos con la misma consulta en SQL Server, y analizar eficiencia, escalabilidad y adecuación de la solución Big Data.

**Estructura del notebook**
1. Entorno y carga de datos
2. Perfil del dataset
3. Operaciones con RDD, DataFrame y Spark SQL (con verificación de resultados)
4. Tabla comparativa de tiempos Spark vs SQL Server
5. Prueba de escalabilidad (volumen creciente)
6. Análisis de eficiencia, escalabilidad y adecuación

## 1. Entorno y carga de datos
Este notebook corre en **Google Colab (capa gratuita)** o en local (Docker `jupyter/pyspark-notebook`). La celda siguiente instala PySpark solo si hace falta.

In [ ]:
import sys, subprocess
try:
    import pyspark
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])
    import pyspark
print("PySpark", pyspark.__version__)

In [ ]:
# --- RUTA DEL CSV -----------------------------------------------------------
# En Colab: sube el archivo (panel de archivos de la izquierda) o ejecuta:
#   from google.colab import files; files.upload()
# En Docker/local: deja el CSV junto al notebook.
RUTA_CSV = "vacunas_covid.csv"

In [ ]:
import time, os, statistics
from functools import reduce
import pandas as pd
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .master("local[*]")                      # usa todos los núcleos disponibles
         .appName("DataSalud_Peru_Spark")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "8")   # valor razonable para modo local
         .getOrCreate())
sc = spark.sparkContext
print("Núcleos (paralelismo por defecto):", sc.defaultParallelism)
print("Spark", spark.version)

In [ ]:
# Lectura del CSV (comillas manejadas: GRUPO_RIESGO contiene comas, p. ej. '60 AÑOS A MÁS, ADULTO MAYOR')
t0 = time.perf_counter()
df = (spark.read.option("header", True).option("inferSchema", True)
      .option("quote", '"').option("escape", '"')
      .csv(RUTA_CSV))
n_filas = df.count()
t_lectura = time.perf_counter() - t0
print(f"Filas: {n_filas:,} | Columnas: {len(df.columns)} | Lectura + conteo: {t_lectura:.2f} s")
print("Particiones del DataFrame:", df.rdd.getNumPartitions())
df.printSchema()
df.createOrReplaceTempView("vacunas_covid")

## 2. Perfil del dataset (estadística descriptiva y calidad)

In [ ]:
df.select("DOSIS", "EDAD").describe().show()
print("Valores nulos por columna:")
nulos = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).toPandas().T
nulos.columns = ["nulos"]
nulos["% nulos"] = (nulos["nulos"] / n_filas * 100).round(2)
display(nulos) if "display" in globals() else print(nulos)

## 3. Operaciones con RDD, DataFrame y Spark SQL
Se ejecutan **las mismas tres operaciones** con los tres objetos, y se verifica que den el mismo resultado:

| Op. | Tipo | Descripción | Equivalente en SQL Server |
|---|---|---|---|
| A | Estadística descriptiva | COUNT, SUM, AVG, MIN y MAX de `DOSIS` (todo el dataset) | `SELECT COUNT(*), SUM(DOSIS), AVG(CAST(DOSIS AS FLOAT)), MIN(DOSIS), MAX(DOSIS) FROM vacunas_covid` |
| B | Agregación | Total de dosis y registros por `FABRICANTE` | `GROUP BY FABRICANTE` |
| C | Agregación + filtro | Consulta crítica del informe 1.3: por provincia de **LA LIBERTAD**, COUNT, SUM y AVG de dosis | `WHERE DEPARTAMENTO='LA LIBERTAD' GROUP BY PROVINCIA` |

In [ ]:
def medir(fn, repeticiones=5):
    """Ejecuta fn una vez de calentamiento (no se cuenta) y luego `repeticiones` veces.
    Devuelve (resultado, media_ms, minimo_ms, desv_ms)."""
    resultado = fn()                                   # calentamiento
    tiempos = []
    for _ in range(repeticiones):
        t = time.perf_counter(); resultado = fn(); tiempos.append((time.perf_counter() - t) * 1000)
    return resultado, statistics.mean(tiempos), min(tiempos), (statistics.stdev(tiempos) if len(tiempos) > 1 else 0.0)

resultados, tiempos = {}, []

def registrar(op, api, fn, rep=5):
    res, media, minimo, desv = medir(fn, rep)
    resultados[(op, api)] = res
    tiempos.append({"Operación": op, "API": api, "Media (ms)": round(media, 1),
                    "Mín (ms)": round(minimo, 1), "Desv (ms)": round(desv, 1)})
    print(f"{op:<3} {api:<10} media={media:9.1f} ms  min={minimo:9.1f} ms")

In [ ]:
# ============ RDD ============
# Se parte de df.rdd para no parsear CSV a mano (hay comas dentro de campos entre comillas).
rdd_dosis = df.select("DOSIS").rdd.map(lambda r: r[0]).filter(lambda x: x is not None)
rdd_fab   = df.select("FABRICANTE", "DOSIS").rdd
rdd_ll    = df.select("DEPARTAMENTO", "PROVINCIA", "DOSIS").rdd

def A_rdd():
    s = rdd_dosis.stats()                              # StatCounter: count, sum, mean, min, max
    return (s.count(), s.sum(), round(s.mean(), 6), s.min(), s.max())

def B_rdd():
    r = (rdd_fab.map(lambda r: (r[0] if r[0] is not None else "SIN FABRICANTE", (1, r[1])))
                .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])).collect())
    return sorted((k, v[0], v[1]) for k, v in r)

def C_rdd():
    r = (rdd_ll.filter(lambda r: r[0] == "LA LIBERTAD")
               .map(lambda r: (r[1], (1, r[2])))
               .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])).collect())
    return sorted((k, v[0], v[1], round(v[1] / v[0], 6)) for k, v in r)

registrar("A", "RDD", A_rdd); registrar("B", "RDD", B_rdd); registrar("C", "RDD", C_rdd)

In [ ]:
# ============ DataFrame ============
def A_df():
    r = df.agg(F.count("*"), F.sum("DOSIS"), F.avg("DOSIS"), F.min("DOSIS"), F.max("DOSIS")).first()
    return (r[0], r[1], round(r[2], 6), r[3], r[4])

def B_df():
    r = (df.withColumn("FABRICANTE", F.coalesce("FABRICANTE", F.lit("SIN FABRICANTE")))
           .groupBy("FABRICANTE").agg(F.count("*").alias("n"), F.sum("DOSIS").alias("s")).collect())
    return sorted((x[0], x[1], x[2]) for x in r)

def C_df():
    r = (df.filter(F.col("DEPARTAMENTO") == "LA LIBERTAD").groupBy("PROVINCIA")
           .agg(F.count("*").alias("n"), F.sum("DOSIS").alias("s"), F.avg("DOSIS").alias("a"))
           .orderBy("PROVINCIA").collect())
    return [(x[0], x[1], x[2], round(x[3], 6)) for x in r]

registrar("A", "DataFrame", A_df); registrar("B", "DataFrame", B_df); registrar("C", "DataFrame", C_df)

In [ ]:
# ============ Spark SQL ============
def A_sql():
    r = spark.sql("SELECT COUNT(*), SUM(DOSIS), AVG(DOSIS), MIN(DOSIS), MAX(DOSIS) FROM vacunas_covid").first()
    return (r[0], r[1], round(r[2], 6), r[3], r[4])

def B_sql():
    r = spark.sql("""SELECT COALESCE(FABRICANTE,'SIN FABRICANTE') AS f, COUNT(*) n, SUM(DOSIS) s
                     FROM vacunas_covid GROUP BY COALESCE(FABRICANTE,'SIN FABRICANTE')""").collect()
    return sorted((x[0], x[1], x[2]) for x in r)

def C_sql():
    r = spark.sql("""SELECT PROVINCIA, COUNT(*) n, SUM(DOSIS) s, AVG(DOSIS) a
                     FROM vacunas_covid WHERE DEPARTAMENTO = 'LA LIBERTAD'
                     GROUP BY PROVINCIA ORDER BY PROVINCIA""").collect()
    return [(x[0], x[1], x[2], round(x[3], 6)) for x in r]

registrar("A", "Spark SQL", A_sql); registrar("B", "Spark SQL", B_sql); registrar("C", "Spark SQL", C_sql)

In [ ]:
# Verificación: los tres objetos deben dar exactamente el mismo resultado
for op in ["A", "B", "C"]:
    r1, r2, r3 = resultados[(op, "RDD")], resultados[(op, "DataFrame")], resultados[(op, "Spark SQL")]
    print(f"Operación {op}: RDD == DataFrame == Spark SQL ->", r1 == r2 == r3)
print("\nA (count, sum, avg, min, max):", resultados[("A", "DataFrame")])
print("\nB (fabricante, registros, dosis):"); [print("  ", x) for x in resultados[("B", "DataFrame")]]
print("\nC (provincia, registros, dosis, promedio) – LA LIBERTAD:"); [print("  ", x) for x in resultados[("C", "DataFrame")]]

In [ ]:
# Plan de ejecución (útil para explicar Catalyst en el informe)
spark.sql("SELECT PROVINCIA, COUNT(*), SUM(DOSIS), AVG(DOSIS) FROM vacunas_covid "
          "WHERE DEPARTAMENTO='LA LIBERTAD' GROUP BY PROVINCIA").explain()

## 4. Comparación de tiempos: Spark vs SQL Server
**Cómo obtener los tiempos de SQL Server** (misma máquina, mismos datos):
1. Importa `vacunas_covid.csv` a SQL Server como tabla `dbo.vacunas_covid` (SSMS → clic derecho en la base → *Tasks → Import Flat File*; `DOSIS` como `INT`).
2. Ejecuta cada consulta **6 veces** con `SET STATISTICS TIME ON;`, descarta la primera (calentamiento) y promedia el **elapsed time** de la pestaña *Messages*.
3. Escribe los promedios (en ms) en el diccionario de la siguiente celda.

```sql
SET STATISTICS TIME ON;
-- A
SELECT COUNT(*), SUM(DOSIS), AVG(CAST(DOSIS AS FLOAT)), MIN(DOSIS), MAX(DOSIS) FROM dbo.vacunas_covid;
-- B
SELECT ISNULL(FABRICANTE,'SIN FABRICANTE') AS FABRICANTE, COUNT(*) AS n, SUM(DOSIS) AS s
FROM dbo.vacunas_covid GROUP BY ISNULL(FABRICANTE,'SIN FABRICANTE');
-- C
SELECT PROVINCIA, COUNT(*) AS n, SUM(DOSIS) AS s, AVG(CAST(DOSIS AS FLOAT)) AS a
FROM dbo.vacunas_covid WHERE DEPARTAMENTO = 'LA LIBERTAD' GROUP BY PROVINCIA ORDER BY PROVINCIA;
SET STATISTICS TIME OFF;
```

In [ ]:
# >>> COMPLETA con tus mediciones de SQL Server (milisegundos, promedio de 5 ejecuciones) <<<
SQL_SERVER_MS = {
    "A": None,   # ej.: 85.0
    "B": None,
    "C": None,
}

In [ ]:
tabla = pd.DataFrame(tiempos)
piv = tabla.pivot(index="Operación", columns="API", values="Media (ms)")[["RDD", "DataFrame", "Spark SQL"]]
piv["SQL Server"] = pd.Series(SQL_SERVER_MS)
if piv["SQL Server"].notna().all():
    piv["Spark SQL / SQL Server (x)"] = (piv["Spark SQL"] / piv["SQL Server"]).round(2)
    piv["DataFrame / SQL Server (x)"] = (piv["DataFrame"] / piv["SQL Server"]).round(2)
else:
    print("⚠ Falta completar SQL_SERVER_MS para calcular las razones Spark/SQL Server.")
print(f"Tiempo de lectura inicial del CSV + conteo en Spark: {t_lectura:.2f} s")
piv

In [ ]:
import matplotlib.pyplot as plt
ax = piv[["RDD", "DataFrame", "Spark SQL"] + (["SQL Server"] if piv["SQL Server"].notna().all() else [])].plot.bar(figsize=(8, 4), rot=0)
ax.set_ylabel("Tiempo medio (ms)"); ax.set_title(f"Tiempo de ejecución por operación ({n_filas:,} filas)")
plt.tight_layout(); plt.savefig("comparativa_tiempos.png", dpi=150); plt.show()

## 5. Prueba de escalabilidad (volumen creciente)
Se replica el dataset **k = 1, 2, 4, 8 veces** (≈1.05 M → ≈8.4 M filas, del orden de los 4.5 M de registros del informe 1.1 y más) y se mide la misma agregación. Si el tiempo crece **menos que proporcionalmente** al volumen, la solución escala bien.

In [ ]:
def df_replicado(k):
    return reduce(DataFrame.unionAll, [df] * k)

def agregacion(d):
    return (d.groupBy("DEPARTAMENTO", "FABRICANTE")
             .agg(F.count("*"), F.sum("DOSIS"), F.avg("DOSIS")).collect())

esc = []
for k in [1, 2, 4, 8]:
    d = df_replicado(k)
    filas = d.count()
    _, media, minimo, _ = medir(lambda: agregacion(d), repeticiones=3)
    esc.append({"Factor k": k, "Filas": filas, "Tiempo medio (ms)": round(media, 1)})
    print(f"k={k}  filas={filas:>10,}  media={media:9.1f} ms")
esc = pd.DataFrame(esc)
base = esc.loc[0, "Tiempo medio (ms)"]
esc["Crecimiento del tiempo (x)"] = (esc["Tiempo medio (ms)"] / base).round(2)
esc["Crecimiento del volumen (x)"] = esc["Factor k"]
esc["Filas/seg"] = (esc["Filas"] / (esc["Tiempo medio (ms)"] / 1000)).round(0).astype(int)
esc

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(esc["Filas"] / 1e6, esc["Tiempo medio (ms)"], marker="o")
ax[0].set_xlabel("Millones de filas"); ax[0].set_ylabel("Tiempo medio (ms)"); ax[0].set_title("Tiempo vs. volumen")
ax[1].plot(esc["Factor k"], esc["Crecimiento del tiempo (x)"], marker="o", label="Tiempo medido")
ax[1].plot(esc["Factor k"], esc["Factor k"], "--", label="Escalamiento lineal (ideal)")
ax[1].set_xlabel("Factor de volumen"); ax[1].set_ylabel("Crecimiento (x)"); ax[1].legend(); ax[1].set_title("Escalabilidad")
plt.tight_layout(); plt.savefig("escalabilidad.png", dpi=150); plt.show()

In [ ]:
# Efecto del paralelismo (núcleos) sobre el dataset base: compara particiones
res_part = []
for p in [1, 2, 4, 8]:
    d = df.repartition(p)
    d.count()
    _, media, _, _ = medir(lambda: agregacion(d), repeticiones=3)
    res_part.append({"Particiones": p, "Tiempo medio (ms)": round(media, 1)})
pd.DataFrame(res_part)

## 6. Análisis de eficiencia, escalabilidad y adecuación de la solución Big Data
> Completa/ajusta los **[corchetes]** con tus números reales después de ejecutar todas las celdas.

**Naturaleza del dataset.** Datos tabulares estructurados (14 columnas, `DOSIS` entera, 1.05 M de filas, ~119 MB en CSV). Es un volumen que **cabe en la memoria de una sola máquina**; el cuello de botella no es el almacenamiento sino el cómputo agregado.

**Eficiencia.** Con el volumen base, Spark SQL/DataFrame resolvió las operaciones A, B y C en **[X] ms**, frente a **[Y] ms** de SQL Server. Spark paga un costo fijo de arranque (JVM, planificación, shuffle) y la lectura inicial del CSV tomó **[Z] s**, mientras que SQL Server consulta datos ya cargados y con páginas en caché. Los RDD fueron **[más lentos/similares]** que DataFrame/SQL porque no se benefician del optimizador Catalyst ni de Tungsten (ver `explain()`).

**Escalabilidad.** Al pasar de 1× a 8× el volumen, el tiempo creció **[W]×** (frente a 8× de crecimiento de datos), es decir, **[sub-lineal / aproximadamente lineal]**. El costo fijo se diluye al crecer el volumen y Spark reparte el trabajo en particiones. SQL Server escala **verticalmente** (más CPU/RAM/disco en un solo servidor), mientras que Spark escala **horizontalmente** (añadiendo nodos al clúster sin cambiar el código).

**Adecuación.** Para el volumen y la naturaleza del dataset actual, **[SQL Server resulta suficiente/más rápido en consultas puntuales; Spark se justifica por…]**. Spark es adecuado cuando: (1) el volumen supera la memoria/almacenamiento de un solo servidor, (2) se integran fuentes heterogéneas (CSV, JSON de RENIPRESS/MongoDB, Parquet), (3) se requiere procesamiento por lotes recurrente o *machine learning* (MLlib), o (4) la DIRESA consolide los registros de los 312 establecimientos y de varios años. No es la mejor opción para consultas transaccionales y de auditoría, donde se mantiene SQL Server (arquitectura híbrida del informe 1.4).

**Calidad de datos detectada.** El **56.04 %** de los registros no tiene `FABRICANTE` (587,609 de 1,048,575). Se trató como categoría `SIN FABRICANTE` y no se descartó.

**Conclusión.** [Redacta 3–4 líneas con tus resultados: qué fue más rápido hoy, cómo escala cada enfoque y qué recomiendas para la DIRESA.]